<a href="https://colab.research.google.com/github/IvanovaKaryna/123/blob/main/%D0%9B%D0%A09_%D0%86%D0%B2%D0%B0%D0%BD%D0%BE%D0%B2%D0%B0_3_10.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Лабораторна робота № 9
## Рекомендаційні системи: SVD та KNNBasic

**ПІБ студента:** Іванова Карина

**Група:** ФІТ 3-10

**Дата:** 09.11.2026

> **Призначення:** шаблон для самостійного виконання ЛР на основі прикладу з алгоритмами `SVD` і `KNNBasic` у бібліотеці `Surprise`. Комірки з позначкою **TODO** потрібно заповнити власним кодом. Це не готовий розв’язок і не заміна офіційної умови роботи.

**Мета:** завантажити дані про оцінки фільмів, побудувати дві рекомендаційні моделі, підібрати гіперпараметри, порівняти моделі за RMSE/MAE та сформувати топ-10 рекомендацій для користувача.

**Дані:** вбудований набір `MovieLens 100K` (`ml-100k`) із бібліотеки `Surprise`.

**Послідовність роботи:** підготовка середовища → дані → розбиття → налаштування моделей → навчання → тестування → рекомендації → висновки.

## 1. Підготовка середовища

Комбінація бібліотек для **Google Colab із Python 3.13**. Встановлення без `--force-reinstall`, щоб без потреби не перевстановлювати всі залежності. Після зміни NumPy чи SciPy у вже запущеному середовищі **перезапустіть сесію** (`Runtime → Restart session`), а потім виконайте комірки нижче з початку.

In [1]:
%pip install -q \
    numpy==2.2.6 \
    scipy==1.15.3 \
    scikit-learn==1.6.1 \
    pandas==2.2.3 \
    scikit-surprise==1.1.5

In [2]:
import sys
import numpy as np
import scipy
import sklearn
import pandas as pd
import surprise

from surprise import Dataset, Reader, SVD, KNNBasic, accuracy
from surprise.model_selection import GridSearchCV
from sklearn.model_selection import train_test_split as sklearn_train_test_split

print('Python:', sys.version.split()[0])
print('NumPy:', np.__version__)
print('SciPy:', scipy.__version__)
print('scikit-learn:', sklearn.__version__)
print('pandas:', pd.__version__)
print('Surprise:', surprise.__version__)

Python: 3.13.18
NumPy: 2.2.6
SciPy: 1.15.3
scikit-learn: 1.6.1
pandas: 2.2.3
Surprise: 1.1.5


## 2. Завантаження та первинний аналіз даних

1. Завантажте вбудований набір `ml-100k`.
2. Перетворіть оцінки у `pandas.DataFrame` з колонками `user_id`, `item_id`, `rating`, `timestamp`.
3. Виведіть розмір таблиці, декілька рядків, кількість унікальних користувачів та фільмів, розподіл оцінок.

**Зверніть увагу:** ідентифікатори користувачів і фільмів у `Surprise` є зовнішніми (raw) ID; вони можуть мати рядковий тип.

In [3]:
# Завантаження даних (готова частина шаблону)
data = Dataset.load_builtin('ml-100k', prompt=False)

ratings_df = pd.DataFrame(
    data.raw_ratings,
    columns=['user_id', 'item_id', 'rating', 'timestamp']
)

print('Розмір:', ratings_df.shape)
display(ratings_df.head())

Розмір: (100000, 4)


,user_id,item_id,rating,timestamp
0,196,242,3.0,881250949
1,186,302,3.0,891717742
2,22,377,1.0,878887116
3,244,51,2.0,880606923
4,166,346,1.0,886397596


In [4]:
# TODO 2.1 — Дослідіть набір даних:
# - кількість унікальних користувачів;
# - кількість унікальних фільмів;
# - мінімальне, максимальне, середнє значення rating;
# - пропущені значення (за наявності).

# Ваш код:
print('Кількість унікальних користувачів:', ratings_df['user_id'].nunique())

print('Кількість унікальних фільмів:', ratings_df['item_id'].nunique())

print('Мінімальна оцінка:', ratings_df['rating'].min())
print('Максимальна оцінка:', ratings_df['rating'].max())
print('Середня оцінка:', ratings_df['rating'].mean())

print('\nКількість пропущених значень:')
print(ratings_df.isnull().sum())

print('\nРозподіл оцінок:')
print(ratings_df['rating'].value_counts().sort_index())

Кількість унікальних користувачів: 943
Кількість унікальних фільмів: 1682
Мінімальна оцінка: 1.0
Максимальна оцінка: 5.0
Середня оцінка: 3.52986

Кількість пропущених значень:
user_id      0
item_id      0
rating       0
timestamp    0
dtype: int64

Розподіл оцінок:
rating
1.0     6110
2.0    11370
3.0    27145
4.0    34174
5.0    21201
Name: count, dtype: int64


**Короткі спостереження щодо даних:**

Я завантажила набір даних MovieLens 100K і переглянула його основну інформацію. У ньому є 100 000 оцінок від 943 користувачів для 1682 фільмів. Оцінки виставлені від 1 до 5. Також я перевірила дані та побачила, що пропущених значень немає. Тому дані підходять для побудови рекомендаційних моделей.

## 3. Розбиття на навчальну та тестову вибірки

Виконайте випадкове розбиття **80% / 20%** із `random_state=42` за прикладом. Для налаштування гіперпараметрів використовуйте **лише навчальну** частину. Тестова вибірка має залишатися недоторканою до фінальної оцінки.

In [5]:
# TODO 3.1 — Створіть train_df та test_df із ratings_df.
# Використайте sklearn_train_test_split(..., test_size=0.2, random_state=42).

# Ваш код:
train_df, test_df = sklearn_train_test_split(
    ratings_df,
    test_size=0.2,
    random_state=42
)

print('Розмір навчальної вибірки:', train_df.shape)
print('Розмір тестової вибірки:', test_df.shape)

Розмір навчальної вибірки: (80000, 4)
Розмір тестової вибірки: (20000, 4)


In [6]:
# TODO 3.2 — Підготуйте об’єкти для Surprise:
# 1. reader = Reader(rating_scale=...)
#    Підказка: data.reader.rating_scale
# 2. train_data = Dataset.load_from_df(...)
#    Візьміть тільки user_id, item_id, rating із train_df.
# 3. testset = список кортежів (user_id, item_id, rating) із test_df.
# 4. Перевірте розміри train/test.

# Ваш код:
reader = Reader(rating_scale=data.reader.rating_scale)

train_data = Dataset.load_from_df(
    train_df[['user_id', 'item_id', 'rating']],
    reader
)

testset = list(
    test_df[['user_id', 'item_id', 'rating']].itertuples(
        index=False,
        name=None
    )
)

print('Розмір train_data:', len(train_df))
print('Розмір testset:', len(testset))
print('Діапазон оцінок:', reader.rating_scale)

Розмір train_data: 80000
Розмір testset: 20000
Діапазон оцінок: (1, 5)


## 4. Гіперпараметри моделей

Для `SVD` розгляньте кількість прихованих факторів, швидкість навчання та регуляризацію. Для `KNNBasic` — число сусідів, мінімальне число сусідів та метрику схожості.

Нижче — **початкова сітка** з аналізованого прикладу. Її можна змінити й обґрунтувати вибір.

In [7]:
# Стартові сітки параметрів (можна змінювати)
param_grid_svd = {
    'n_factors': [50, 100],
    'lr_all': [0.005, 0.010],
    'reg_all': [0.02, 0.05]
}

param_grid_knn = {
    'k': [20, 40],
    'min_k': [1, 5],
    'sim_options': {
        'name': ['cosine', 'pearson'],
        'user_based': [False, True]
    }
}

**Обґрунтування вибору параметрів:**

Для моделі SVD я обрала два значення кількості прихованих факторів - 50 і 100. Також перевірила різні значення швидкості навчання та регуляризації, щоб знайти найкращі налаштування моделі. Для KNNBasic я використала 20 і 40 сусідів, мінімальну кількість сусідів 1 і 5 та дві метрики схожості cosine і pearson. Так я змогла порівняти різні налаштування обох моделей і визначити, які з них дають кращий результат.

## 5. Пошук найкращих гіперпараметрів (GridSearchCV)

Для обох моделей використайте `GridSearchCV` з метриками `rmse` та `mae` і `cv=3`. Навчайте об’єкти пошуку **лише на `train_data`**.

In [8]:
# TODO 5.1 — Підберіть гіперпараметри SVD.
# Підказка: GridSearchCV(SVD, param_grid_svd,
#                        measures=['rmse', 'mae'], cv=3, n_jobs=-1)
# Після створення викличте .fit(train_data).

# Ваш код:
grid_search_svd = GridSearchCV(
    SVD,
    param_grid_svd,
    measures=['rmse', 'mae'],
    cv=3,
    n_jobs=-1
)

grid_search_svd.fit(train_data)

print('Пошук параметрів SVD завершено.')

Пошук параметрів SVD завершено.


In [9]:
# TODO 5.2 — Підберіть гіперпараметри KNNBasic.
# Підказка: використайте param_grid_knn та ті самі метрики/cv.

# Ваш код:
grid_search_knn = GridSearchCV(
    KNNBasic,
    param_grid_knn,
    measures=['rmse', 'mae'],
    cv=3,
    n_jobs=-1
)

grid_search_knn.fit(train_data)

print('Пошук параметрів KNNBasic завершено.')

Пошук параметрів KNNBasic завершено.


In [10]:
# TODO 5.3 — Виведіть найкращий RMSE, MAE та параметри обох моделей.
# Підказки:
# grid_search_svd.best_score['rmse']
# grid_search_svd.best_params['rmse']
# grid_search_knn.best_score['rmse']

# Ваш код:
print('SVD')
print('Найкращий RMSE:', grid_search_svd.best_score['rmse'])
print('Найкращий MAE:', grid_search_svd.best_score['mae'])
print('Параметри за RMSE:', grid_search_svd.best_params['rmse'])

print('\nKNNBasic')
print('Найкращий RMSE:', grid_search_knn.best_score['rmse'])
print('Найкращий MAE:', grid_search_knn.best_score['mae'])
print('Параметри за RMSE:', grid_search_knn.best_params['rmse'])

SVD
Найкращий RMSE: 0.9415042699475554
Найкращий MAE: 0.7424274957454878
Параметри за RMSE: {'n_factors': 50, 'lr_all': 0.01, 'reg_all': 0.05}

KNNBasic
Найкращий RMSE: 1.0261481632098697
Найкращий MAE: 0.8141970445511507
Параметри за RMSE: {'k': 40, 'min_k': 1, 'sim_options': {'name': 'cosine', 'user_based': True}}


**Аналіз результатів кросвалідації:**
Модель | Найкращий CV RMSE | CV MAE | Найкращі параметри за RMSE |
|:--|--:|--:|:--|
| SVD | 0.94233 | 0.74352 | n_factors=50, lr_all=0.01, reg_all=0.05 |
| KNNBasic | 1.02735 | 0.81506 | k=40, min_k=1, cosine, user_based=True |

Після перевірки результатів я побачила, що модель SVD показала кращі результати, ніж KNNBasic. Для SVD значення RMSE становить 0.94233, а MAE - 0.74352. Для KNNBasic отримано RMSE 1.02735 та MAE 0.81506. Найкращими параметрами для SVD стали n_factors=50, lr_all=0.01 та reg_all=0.05. Для KNNBasic найкраще спрацювали k=40, min_k=1, метрика cosine та user_based=True. Отже, я обрала модель SVD, оскільки вона має менші похибки та точніше прогнозує оцінки користувачів.

## 6. Навчання остаточних моделей

Побудуйте `trainset = train_data.build_full_trainset()`. Створіть нові екземпляри SVD і KNNBasic з параметрами, обраними за **CV RMSE**, після чого навчіть їх на **всій навчальній вибірці**.

In [11]:
# TODO 6.1 — Отримайте best_svd_params і best_knn_params за CV RMSE.
# TODO 6.2 — Створіть trainset із train_data.
# TODO 6.3 — Навчіть best_svd і best_knn.
# Для відтворюваності задайте random_state=42 для SVD.

# Ваш код:
best_svd_params = grid_search_svd.best_params['rmse']
best_knn_params = grid_search_knn.best_params['rmse']

print('Параметри SVD:', best_svd_params)
print('Параметри KNNBasic:', best_knn_params)

trainset = train_data.build_full_trainset()

best_svd = SVD(
    **best_svd_params,
    random_state=42
)

best_knn = KNNBasic(
    **best_knn_params
)

best_svd.fit(trainset)
best_knn.fit(trainset)

print('\nОбидві моделі успішно навчені.')

Параметри SVD: {'n_factors': 50, 'lr_all': 0.01, 'reg_all': 0.05}
Параметри KNNBasic: {'k': 40, 'min_k': 1, 'sim_options': {'name': 'cosine', 'user_based': True}}
Computing the cosine similarity matrix...
Done computing similarity matrix.

Обидві моделі успішно навчені.


## 7. Оцінювання на незалежній тестовій вибірці

Отримайте передбачення для `testset` методами `.test()` обох моделей. Обчисліть `RMSE` та `MAE` за допомогою `surprise.accuracy`.

**Методична примітка:** вибирайте модель для рекомендацій за результатами **кросвалідації**, а незалежну тестову вибірку використовуйте для фінальної оцінки якості, а не для повторного підбору.

In [12]:
# TODO 7.1 — Отримайте svd_predictions та knn_predictions.
# TODO 7.2 — Обчисліть svd_rmse, svd_mae, knn_rmse, knn_mae.
# Підказка: accuracy.rmse(predictions, verbose=False)
#           accuracy.mae(predictions, verbose=False)

# Ваш код:
svd_predictions = best_svd.test(testset)
knn_predictions = best_knn.test(testset)

svd_rmse = accuracy.rmse(svd_predictions, verbose=False)
svd_mae = accuracy.mae(svd_predictions, verbose=False)

knn_rmse = accuracy.rmse(knn_predictions, verbose=False)
knn_mae = accuracy.mae(knn_predictions, verbose=False)

print('Результати SVD')
print('RMSE:', svd_rmse)
print('MAE:', svd_mae)

print('\nРезультати KNNBasic')
print('RMSE:', knn_rmse)
print('MAE:', knn_mae)

Результати SVD
RMSE: 0.9191746414249822
MAE: 0.7233554064545319

Результати KNNBasic
RMSE: 1.0119527444069378
MAE: 0.7984207578945732


In [13]:
# TODO 7.3 — Сформуйте DataFrame results із колонками:
# Model, RMSE, MAE.
# Відсортуйте за RMSE та покажіть порівняння.

# Ваш код:
results = pd.DataFrame({
    'Model': ['SVD', 'KNNBasic'],
    'RMSE': [svd_rmse, knn_rmse],
    'MAE': [svd_mae, knn_mae]
})

results = results.sort_values(by='RMSE').reset_index(drop=True)

display(results)

,Model,RMSE,MAE
0,SVD,0.919175,0.723355
1,KNNBasic,1.011953,0.798421


**Таблиця тестових результатів:**

| Модель | Test RMSE | Test MAE |
|:--|--:|--:|
| SVD | 0.919175 | 0.723355 |
| KNNBasic | 1.011953 | 0.798421 |

**Порівняння з CV:**

Я порівняла результати кросвалідації та тестування обох моделей. Для SVD значення RMSE змінилося з 0.94233 до 0.91918, а MAE з 0.74352 до 0.72336. Для KNNBasic RMSE змінилося з 1.02735 до 1.01195, а MAE з 0.81506 до 0.79842. Результати вийшли досить близькими. На тестовій вибірці похибки обох моделей трохи зменшилися. Після порівняння я зробила висновок, що SVD працює краще, оскільки має менші значення похибок.

## 8. Топ-10 рекомендацій для користувача

1. Оберіть модель із кращим **CV RMSE**.
2. Задайте `user_id` (вихідний приклад використовує `'196'`).
3. Отримайте фільми, які цей користувач уже оцінив у навчальній вибірці.
4. Для решти фільмів обчисліть прогнозований рейтинг `.predict(uid=..., iid=...).est`.
5. Виведіть **10 фільмів із найбільшим прогнозованим рейтингом**.

**Важливо:** не пропонуйте користувачу фільми, які він уже оцінив у навчальних даних. Для коректного перетворення внутрішніх та зовнішніх ID використовуйте `trainset.to_inner_uid()`, `trainset.to_raw_iid()`.

In [14]:
# Початковий ID можна змінити
user_id = '196'

# TODO 8.1 — Визначте найкращу за CV RMSE модель.
# TODO 8.2 — Перевірте, що користувач присутній у trainset.
# TODO 8.3 — Знайдіть неоцінені ним фільми.
# TODO 8.4 — Спрогнозуйте оцінки та покажіть топ-10.
# Бажані колонки результату: item_id, predicted_rating.
# Передбачте ситуацію, коли обраного user_id немає у trainset.

# Ваш код:
if grid_search_svd.best_score['rmse'] <= grid_search_knn.best_score['rmse']:
    best_model = best_svd
    model_name = 'SVD'
else:
    best_model = best_knn
    model_name = 'KNNBasic'

print('Обрана модель:', model_name)

try:
    inner_uid = trainset.to_inner_uid(user_id)

    rated_items = {
        inner_iid for inner_iid, rating in trainset.ur[inner_uid]
    }

    all_items = [
        trainset.to_raw_iid(inner_iid)
        for inner_iid in trainset.all_items()
    ]

    unrated_items = [
        iid for iid in all_items
        if trainset.to_inner_iid(iid) not in rated_items
    ]

    recommendations = []

    for iid in unrated_items:
        predicted_rating = best_model.predict(user_id, iid).est

        recommendations.append({
            'item_id': iid,
            'predicted_rating': predicted_rating
        })

    recommendations_df = pd.DataFrame(recommendations)

    recommendations_df = recommendations_df.sort_values(
        by='predicted_rating',
        ascending=False
    ).head(10).reset_index(drop=True)

    print('\nТоп-10 рекомендацій для користувача', user_id)
    display(recommendations_df)

except ValueError:
    print('Користувача', user_id, 'немає у навчальній вибірці.')

Обрана модель: SVD

Топ-10 рекомендацій для користувача 196


,item_id,predicted_rating
0,320,4.490358
1,64,4.470680
2,480,4.413710
3,313,4.401920
4,513,4.399624
5,483,4.398092
6,318,4.392102
7,656,4.385379
8,22,4.377388
9,272,4.376010


## 9. Перевірка виконання

Позначте після реалізації:

- [+] Дані `ml-100k` завантажено та перевірено.
- [+] Є розбиття 80/20 з фіксованим `random_state`.
- [+] `Reader`, `train_data` і `testset` створені коректно.
- [+] GridSearchCV для **обох** моделей виконано лише на `train_data`.
- [+] Для SVD і KNNBasic показано параметри та результати CV.
- [+] Моделі навчено на повній навчальній частині.
- [+] На незалежній тестовій вибірці розраховано **RMSE та MAE** для обох моделей.
- [+] Наведено порівняльну таблицю та інтерпретацію.
- [+] Сформовано топ-10 рекомендацій без уже оцінених фільмів.
- [+] Усі комірки виконуються послідовно після перезапуску середовища.

## 10. Висновки

Заповніть **власними результатами**:

1. Яку модель обрано за кросвалідацією і чому?
2. Які найкращі гіперпараметри отримано для обох моделей?
3. Які підсумкові значення RMSE та MAE на тестовій вибірці?
4. Чи є різниця між результатами CV та тестування?
5. Які обмеження мають отримані рекомендації (наприклад, нові користувачі/фільми)?

**Висновок:** Під час виконання лабораторної роботи я ознайомилася з рекомендаційними системами та навчилася працювати з моделями SVD і KNNBasic. Я завантажила набір даних MovieLens 100K та підготувала його для роботи. Після підбору параметрів з'ясувалося, що модель SVD працює краще за KNNBasic. На тестовій вибірці SVD отримала RMSE = 0.919175 і MAE = 0.723355, а KNNBasic - RMSE = 1.011953 і MAE = 0.798421. За допомогою SVD я сформувала 10 рекомендацій для користувача з ID 196. Також я спробувала власноруч створити алгоритм рекомендацій на основі схожості користувачів. У результаті я отримала практичний досвід роботи з рекомендаційними системами та порівняння їхньої точності.

---
**Примітка до шаблону:** вихідний студентський ноутбук містив блок «Доцільна реалізація» з SVD, KNNBasic і топ-10 рекомендаціями. Тому тут відтворено ці етапи; додаткових офіційних вимог, не наведених у зразку, не припускаємо.

**Додаткове завдання. Написати власний рекомендаційний алгоритм з нуля**

In [15]:
#Створення матриці оцінок
rating_matrix = train_df.pivot_table(
    index='user_id',
    columns='item_id',
    values='rating'
).fillna(0)

#Середні оцінки користувачів
user_means = rating_matrix.replace(0, np.nan).mean(axis=1)
global_mean = train_df['rating'].mean()

#Центрування оцінок
centered_matrix = rating_matrix.sub(user_means, axis=0).where(
    rating_matrix > 0, 0
)

#Обчислення косинусної схожості
matrix = centered_matrix.to_numpy()
norms = np.linalg.norm(matrix, axis=1)
norms[norms == 0] = 1

normalized_matrix = matrix / norms[:, np.newaxis]
similarity_matrix = normalized_matrix @ normalized_matrix.T


#Функція рекомендацій
def custom_recommend(user_id, top_n=10):

    if user_id not in rating_matrix.index:
        print('Користувача немає у навчальній вибірці.')
        return pd.DataFrame()

    user_index = rating_matrix.index.get_loc(user_id)

    similarities = similarity_matrix[user_index].copy()
    similarities[user_index] = 0

    user_ratings = rating_matrix.iloc[user_index]
    unrated_items = user_ratings[user_ratings == 0].index

    predictions = []

    for item_id in unrated_items:

        item_ratings = rating_matrix[item_id]
        rated_users = item_ratings[item_ratings > 0].index

        indexes = rating_matrix.index.get_indexer(rated_users)
        item_similarities = similarities[indexes]

        ratings = item_ratings.loc[rated_users].to_numpy()
        means = user_means.loc[rated_users].to_numpy()

        weights = np.maximum(item_similarities, 0)

        if weights.sum() > 0:
            predicted = user_means.iloc[user_index] + (
                np.sum(weights * (ratings - means)) / weights.sum()
            )
        else:
            predicted = global_mean

        predicted = np.clip(predicted, 1, 5)

        predictions.append({
            'item_id': item_id,
            'predicted_rating': predicted
        })

    result = pd.DataFrame(predictions)

    return result.sort_values(
        by='predicted_rating',
        ascending=False
    ).head(top_n).reset_index(drop=True)


#Формування рекомендацій
custom_recommendations = custom_recommend(user_id, top_n=10)

print('Власний алгоритм рекомендацій для користувача:', user_id)
display(custom_recommendations)

Власний алгоритм рекомендацій для користувача: 196


,item_id,predicted_rating
0,1103,5.000000
1,1554,5.000000
2,1306,5.000000
3,1629,5.000000
4,1628,5.000000
5,1500,5.000000
6,1642,5.000000
7,1398,4.980296
8,1467,4.957341
9,1653,4.871921


У додатковому завданні я самостійно реалізувала алгоритм рекомендацій User-Based Collaborative Filtering. Для цього я використала косинусну схожість, щоб знаходити користувачів із подібними оцінками. На основі їхніх оцінок програма підібрала 10 фільмів для користувача з ID 196. Найвищу прогнозовану оцінку 5.0 - отримали сім фільмів. Таким чином, я навчилася створювати просту рекомендаційну систему без використання готової моделі.